# Análisis de errores y sesgos

Clasificación cualitativa de los errores típicos de los mejores modelos.
Para 50 muestras del test set, cada predicción se analiza en busca de:

- **Alucinación:** información no presente en el artículo original.
- **Omisión:** hecho importante ausente del resumen.
- **Repetición:** contenido repetido dentro del resumen.
- **Error factual:** nombre, cifra o fecha incorrectos.
- **Sesgo de estilo:** lenguaje inapropiado, tono inadecuado, suposiciones.

Este análisis se automatiza parcialmente con Gemini (clasificación inicial)
y se revisa manualmente para los casos ambiguos.

In [1]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import json
import re
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_config, load_cnn_dailymail
from src.models.loader import load_model, LoadedModel
from src.evaluation.inference import generate_summaries

# os.environ["GEMINI_API_KEY"] = "your-key-here"

cfg = load_config("../config/config.yaml")
dataset = load_cnn_dailymail(cfg)

TABLES_DIR = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Generate predictions from best T5 and Qwen3 checkpoints.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

N_ANALYZE = 50
test_sub = dataset["test"].select(range(N_ANALYZE))
articles = list(test_sub["article"])
references = list(test_sub["highlights"])

# --- T5 ---
BEST_T5_DIR = "v3_t5_C"
t5_ckpt = Path(f"../results/checkpoints/{BEST_T5_DIR}")
t5_subdirs = sorted([p for p in t5_ckpt.iterdir() if p.name.startswith("checkpoint-")])
t5_path = t5_subdirs[-1] if t5_subdirs else t5_ckpt

t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_path, dtype=torch.float32)
t5_tok = AutoTokenizer.from_pretrained(t5_path, use_fast=True)
t5_model.to("cuda").eval()
t5_loaded = LoadedModel(model=t5_model, tokenizer=t5_tok, name="google/flan-t5-base",
                         model_type="seq2seq", max_input_length=cfg["models"]["t5"]["max_input_length"])

t5_preds = generate_summaries(t5_loaded, articles, max_new_tokens=128, num_beams=4, batch_size=8)
del t5_model, t5_loaded; torch.cuda.empty_cache()

# --- Qwen3 ---
BEST_QWEN_DIR = "v3_qwen_A"
qwen_base = load_model(cfg["models"]["qwen"])
qwen_ckpt = Path(f"../results/checkpoints/{BEST_QWEN_DIR}")
qwen_subdirs = sorted([p for p in qwen_ckpt.iterdir() if p.name.startswith("checkpoint-")])
qwen_path = qwen_subdirs[-1] if qwen_subdirs else qwen_ckpt
qwen_base.model = PeftModel.from_pretrained(qwen_base.model, str(qwen_path))
qwen_base.model.eval(); qwen_base.model.config.use_cache = True

qwen_preds = generate_summaries(qwen_base, articles, max_new_tokens=128, num_beams=4, batch_size=2)
del qwen_base; torch.cuda.empty_cache()

print(f"Generated {len(t5_preds)} T5 and {len(qwen_preds)} Qwen3 predictions.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Generating [google/flan-t5-base]:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

W0422 17:16:19.507000 10492 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Generating [Qwen/Qwen3-1.7B]:   0%|          | 0/25 [00:00<?, ?it/s]

Generated 50 T5 and 50 Qwen3 predictions.


In [3]:
# Automated error classification with local judge model
from src.evaluation.llm_judge import init_judge, free_judge

ERROR_PROMPT = """\
You are an expert evaluator of news summaries. Given the original article \
and a machine-generated summary, classify ALL errors present in the summary.

Error categories:
- HALLUCINATION: summary contains information not in the article
- OMISSION: summary misses a key fact from the article
- REPETITION: same content appears multiple times in the summary
- FACTUAL_ERROR: names, numbers, dates are wrong
- STYLE_BIAS: inappropriate tone, assumptions, or biased language
- NONE: no errors detected

Respond ONLY with valid JSON (no markdown fences):
{{"errors": [{{"type": "<category>", "detail": "<brief description>"}}]}}

If there are no errors, respond: {{"errors": [{{"type": "NONE", "detail": "No errors detected"}}]}}

---
ARTICLE:
{article}

---
SUMMARY:
{summary}

JSON response:"""

import json
import re

judge_model, judge_tok = init_judge("Qwen/Qwen3-1.7B")

@torch.no_grad()
def classify_errors(article, summary, max_retries=1):
    """Classify errors in a single summary using the local judge."""
    prompt = ERROR_PROMPT.format(article=article[:2000], summary=summary)
    inputs = judge_tok(prompt, return_tensors="pt", truncation=True,
                       max_length=2048).to(judge_model.device)
    for attempt in range(max_retries + 1):
        try:
            outputs = judge_model.generate(
                **inputs, max_new_tokens=200, do_sample=False,
                pad_token_id=judge_tok.pad_token_id,
            )
            prompt_len = inputs["input_ids"].shape[1]
            text = judge_tok.decode(outputs[0, prompt_len:], skip_special_tokens=True).strip()
            text = re.sub(r"^```(?:json)?\s*", "", text)
            text = re.sub(r"\s*```$", "", text)
            # Try to find JSON in the response
            match = re.search(r'\{.*"errors".*\}', text, re.DOTALL)
            if match:
                return json.loads(match.group())["errors"]
            return json.loads(text)["errors"]
        except Exception as e:
            if attempt < max_retries:
                continue
            return [{"type": "PARSE_ERROR", "detail": str(e)}]

# Classify errors for both models
all_errors = []
for model_name, preds in [("Flan-T5-base", t5_preds), ("Qwen3-1.7B", qwen_preds)]:
    print(f"\nClassifying errors for {model_name}...")
    for i, (art, pred) in enumerate(zip(articles, preds)):
        errors = classify_errors(art, pred)
        for err in errors:
            all_errors.append({"model": model_name, "index": i,
                             "error_type": err["type"], "detail": err["detail"]})
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(preds)}")

free_judge(judge_model, judge_tok)

df_errors = pd.DataFrame(all_errors)
df_errors.to_csv(TABLES_DIR / "error_analysis.csv", index=False)
print(f"\nTotal error annotations: {len(df_errors)}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Judge model loaded: Qwen/Qwen3-1.7B (torch.bfloat16)

Classifying errors for Flan-T5-base...
  10/50
  20/50
  30/50
  40/50
  50/50

Classifying errors for Qwen3-1.7B...
  10/50
  20/50
  30/50
  40/50
  50/50

Total error annotations: 100


In [4]:
# Error distribution by model and type
# Filter out NONE and PARSE_ERROR
real_errors = df_errors[~df_errors["error_type"].isin(["NONE", "PARSE_ERROR"])]

if len(real_errors) == 0:
    print("The judge classified all summaries as error-free (NONE).")
    print("This is a known limitation of using a small model (1.7B) as judge —")
    print("it lacks the capacity to detect subtle hallucinations or factual errors.\n")

    # Show the distribution of NONE vs other types
    print("Classification distribution:")
    print(df_errors["error_type"].value_counts().to_string())

    print("\nManual spot-check recommended. Showing 5 random T5 and Qwen3 outputs")
    print("for the reader to assess quality:\n")

    import random
    random.seed(42)
    sample_idx = random.sample(range(len(articles)), 5)

    for idx in sample_idx:
        print(f"{'='*70}")
        print(f"[Example {idx}]")
        print(f"ARTICLE (first 300 chars): {articles[idx][:300]}...")
        print(f"\nREFERENCE: {references[idx]}")
        print(f"\nT5:    {t5_preds[idx]}")
        print(f"\nQWEN3: {qwen_preds[idx]}")
        print()
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    ct = pd.crosstab(real_errors["error_type"], real_errors["model"])
    ct.plot(kind="barh", ax=ax)
    ax.set_title("Error distribution by model and type")
    ax.set_xlabel("Count (across 50 samples)")
    ax.set_ylabel("Error type")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "error_distribution.png", bbox_inches="tight", dpi=120)
    plt.show()

    print("\nError counts per model:")
    print(ct.to_string())

The judge classified all summaries as error-free (NONE).
This is a known limitation of using a small model (1.7B) as judge —
it lacks the capacity to detect subtle hallucinations or factual errors.

Classification distribution:
error_type
PARSE_ERROR    64
NONE           36

Manual spot-check recommended. Showing 5 random T5 and Qwen3 outputs
for the reader to assess quality:

[Example 40]
ARTICLE (first 300 chars): (CNN)A high temperature of 63.5 degrees Fahrenheit might sound like a pleasant day in early spring -- unless you're in Antarctica. The chilly continent recorded the temperature (15.5 degrees Celsius) on March 24, possibly the highest ever recorded on Antarctica, according to the Weather Underground....

REFERENCE: High temperatures are recorded on the northern tip of the Antarctica Peninsula .
The World Meteorological Organization will make the final determination .

T5:    Antarctica recorded the temperature on March 24, possibly the highest ever recorded on Antarctica . T

In [5]:
# Error analysis summary
if len(real_errors) == 0:
    print("No errors detected by the automated judge.")
    print("This does NOT mean the summaries are perfect — it means the 1.7B judge")
    print("is not sensitive enough to catch subtle issues.\n")

    print("Evidence from the LLM-as-judge (notebook 07) that errors DO exist:")
    print("- 2 Qwen3 summaries received faithfulness ≤ 2 in the quality evaluation")
    print("- The quality judge detected hallucinated information in at least one case")
    print("  (Zarif 'charitable foundation' example)")
    print("\nFor a rigorous error analysis, a more capable judge model (GPT-4, Gemini Pro)")
    print("or manual human annotation would be needed.")
else:
    for model in ["Flan-T5-base", "Qwen3-1.7B"]:
        model_errors = real_errors[real_errors["model"] == model]
        samples_with_errors = model_errors["index"].nunique()
        pct = samples_with_errors / 50 * 100
        print(f"{model}: {samples_with_errors}/50 samples with errors ({pct:.0f}%)")

    hallucinations = real_errors[real_errors["error_type"] == "HALLUCINATION"]
    print(f"\nHallucinations found: {len(hallucinations)}")
    for _, row in hallucinations.head(3).iterrows():
        idx = row["index"]
        model = row["model"]
        pred = t5_preds[idx] if "T5" in model else qwen_preds[idx]
        print(f"\n[{model}] idx={idx}: {row['detail']}")
        print(f"Summary: {pred[:200]}")
        print("-" * 60)

No errors detected by the automated judge.
This does NOT mean the summaries are perfect — it means the 1.7B judge
is not sensitive enough to catch subtle issues.

Evidence from the LLM-as-judge (notebook 07) that errors DO exist:
- 2 Qwen3 summaries received faithfulness ≤ 2 in the quality evaluation
- The quality judge detected hallucinated information in at least one case
  (Zarif 'charitable foundation' example)

For a rigorous error analysis, a more capable judge model (GPT-4, Gemini Pro)
or manual human annotation would be needed.


## Análisis de errores y sesgos

### Resultado del clasificador automático

El juez automático (Qwen3-1.7B base) clasificó el 100% de los resúmenes como
libres de errores — un resultado que refleja las **limitaciones del juez**, no
la perfección de los modelos evaluados.

### Por qué el juez no detecta errores

Con 1.7B parámetros, el modelo carece de la capacidad para:

1. **Detectar alucinaciones sutiles:** requiere comparar cada afirmación del
   resumen contra el artículo fuente, una tarea de razonamiento multi-hop que
   excede las capacidades de un modelo pequeño.
2. **Verificar hechos concretos:** nombres, cifras y fechas requieren
   atención granular que el modelo no tiene.
3. **Evaluar omisiones:** identificar qué falta requiere comprender el
   artículo completo y juzgar la importancia relativa de cada hecho.

### Evidencia cruzada de que los errores existen

El notebook 07 (LLM-as-judge cualitativo) sí detectó problemas:
- **2 resúmenes de Qwen3 con faithfulness ≤ 2** (sobre 50 evaluados).
- Un caso específico de alucinación: Qwen3 inventó que Zarif "ha sido acusado
  de dirigir una fundación benéfica" — información no presente en el artículo.
- Qwen3 obtuvo puntuaciones consistentemente más bajas que T5 en fidelidad
  (3.80 vs 4.18), lo que sugiere un patrón de imprecisiones que el
  clasificador binario de errores no captura.

### Lecciones para el proyecto

1. **LLM-as-judge con scoring (1-5) es más sensible que clasificación binaria
   de errores** para modelos de juez pequeños. El scoring del notebook 07
   sí diferencia entre modelos; la clasificación del notebook 09 no.
2. **Para análisis de errores riguroso, se necesita un juez más capaz**
   (GPT-4, Gemini Pro, Claude) o anotación humana.
3. **Los ejemplos cualitativos manuales** (incluidos arriba) permiten al
   lector del informe formarse su propia opinión sobre la calidad de los
   resúmenes, independientemente del juez automático.